In [1]:
from __future__ import annotations

# ============================================================
# 0. IMPORTS
# ============================================================

import operator
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

import requests
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langgraph.checkpoint.postgres import PostgresSaver

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq


# ============================================================
# 1. ENVIRONMENT
# ============================================================

load_dotenv()

# ============================================================
# ENV EXPECTED IN .env
# ============================================================
#
# DATABASE_URL=postgresql://...
# GOOGLE_API_KEY=...
# GROQ_API_KEY=...
#
# Optional:
#
# GEMINI_MAIN_MODEL=gemini-2.5-flash
# GEMINI_FAST_MODEL=gemini-2.5-flash-lite
# GROQ_WORKER_MODEL=llama-3.1-8b-instant
#
# ============================================================


def get_database_url() -> str:
    """
    Lấy DATABASE_URL từ file .env.

    DATABASE_URL dùng để kết nối PostgreSQL,
    nơi LangGraph lưu checkpoint / state của workflow.
    """

    database_url = os.getenv("DATABASE_URL")

    if not database_url:
        raise ValueError(
            "DATABASE_URL is missing. "
            "Please add your Render PostgreSQL External Database URL to .env"
        )

    # PostgreSQL trên Render thường cần SSL.
    # Nếu URL chưa có sslmode thì tự động thêm.
    if "sslmode=" not in database_url:
        separator = "&" if "?" in database_url else "?"
        database_url = f"{database_url}{separator}sslmode=require"

    return database_url


# ============================================================
# 2. PYDANTIC SCHEMAS
# ============================================================
#
# Đây là "hợp đồng dữ liệu" giữa LLM và workflow.
#
# Thay vì để LLM trả JSON tự do:
#
#     {"title": "...", "abc": "..."}
#
# ta ép nó phải tuân theo schema:
#
#     Plan
#       └── tasks
#             ├── Task
#             ├── Task
#             └── ...
#
# Đây là nền tảng để dùng:
#
#     llm.with_structured_output(...)
#
# ============================================================


# ------------------------------------------------------------
# 2.1 Task = Công việc cụ thể phải làm để hoàn thành Plan
# ------------------------------------------------------------

# Task
# │
# ├── id                  → Công việc số mấy?
# ├── title               → Tên phần cần viết?
# ├── goal                → Viết phần này để đạt mục tiêu gì?
# ├── bullets             → Cụ thể phải nói những gì để đạt được mục tiêu?
# ├── target_words        → Viết dài khoảng bao nhiêu?
# ├── tags                → Phần này thuộc chủ đề nào?
# ├── requires_research   → Có cần research không?
# ├── requires_citations  → Có cần nguồn trích dẫn không?
# └── requires_code       → Có cần code example không?

class Task(BaseModel):

    id: int

    title: str

    goal: str = Field(
        ...,
        description=(
            "One sentence describing what the reader "
            "should be able to do/understand after this section."
        ),
    )

    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=6,
        description=(
            "3–6 concrete, non-overlapping subpoints "
            "to cover in this section."
        ),
    )

    target_words: int = Field(
        ...,
        description="Target word count for this section (120–550).",
    )

    tags: List[str] = Field(default_factory=list)

    # Section này có cần thông tin từ web không?
    requires_research: bool = False

    # Section này có bắt buộc citation không?
    requires_citations: bool = False

    # Section này có cần code example không?
    requires_code: bool = False


# ------------------------------------------------------------
# 2.2 Plan = Kế hoạch tổng thể cho bài viết
# ------------------------------------------------------------

# Plan
# │
# ├── blog_title   → Bài viết tên gì?
# ├── audience     → Viết cho ai?
# ├── tone         → Viết như thế nào?
# ├── blog_kind    → Viết loại bài gì?
# ├── constraints  → Toàn bài phải tuân thủ gì?
# └── tasks        → Cần làm những section nào?

class Plan(BaseModel):

    blog_title: str

    audience: str

    tone: str

    blog_kind: Literal[
        "explainer",
        "tutorial",
        "news_roundup",
        "comparison",
        "system_design",
    ] = "explainer"

    constraints: List[str] = Field(default_factory=list)

    tasks: List[Task]


# ------------------------------------------------------------
# 2.3 Evidence = một mẩu bằng chứng/thông tin lấy từ một nguồn cụ thể trên Internet
# ------------------------------------------------------------

# EvidenceItem
# │
# ├── title          → Tiêu đề nguồn
# ├── url            → Link nguồn để kiểm chứng/citation
# ├── published_at   → Ngày xuất bản (có thể không có)
# ├── snippet        → Đoạn tóm tắt/nội dung liên quan
# └── source         → Tên tổ chức/website cung cấp nguồn

class EvidenceItem(BaseModel):

    title: str

    url: str

    # Có thể Tavily không trả published date.
    published_at: Optional[str] = None

    snippet: Optional[str] = None

    source: Optional[str] = None


# ------------------------------------------------------------
# 2.4 RouterDecision = quyết định của Router xem có cần research không, research theo kiểu nào và cần search những gì?
# ------------------------------------------------------------

# RouterDecision
# │
# ├── mode
# │      → Chọn cách xử lý:
# │           closed_book → chỉ dùng kiến thức LLM
# │           hybrid      → LLM + research
# │           open_book   → research bên ngoài
# │
# ├── queries
# │      → Các câu truy vấn cần dùng để search
# │
# └── rationale
#        → Giải thích lý do Router chọn mode đó

class RouterDecision(BaseModel):

    # Có cần research không?
    needs_research: bool

    # closed_book / hybrid / open_book
    mode: Literal[
        "closed_book",
        "hybrid",
        "open_book",
    ]

    # Các query gửi cho Tavily
    queries: List[str] = Field(default_factory=list)


# ------------------------------------------------------------
# 2.5 EvidencePack = dùng để đóng gói (pack) danh sách các EvidenceItem
# ------------------------------------------------------------

# EvidencePack
# │
# └── items
#       → Gom nhiều EvidenceItem thành một danh sách
#       → Đóng gói toàn bộ evidence của quá trình research

class EvidencePack(BaseModel):

    evidence: List[EvidenceItem] = Field(
        default_factory=list
    )


# ------------------------------------------------------------
# 2.6 ImageSpec = bản thiết kế cho 1 hình ảnh trong bài blog
# ------------------------------------------------------------

# ImageSpec
# │
# ├── placeholder → Vị trí chèn ảnh trong Markdown
# ├── filename    → Tên file ảnh được lưu
# ├── alt         → Mô tả ảnh cho accessibility
# ├── caption     → Chú thích hiển thị dưới ảnh
# ├── prompt      → Prompt gửi Image Model
# ├── size        → Kích thước ảnh
# └── quality     → Chất lượng ảnh

class ImageSpec(BaseModel):

    # Placeholder xuất hiện trong Markdown
    #
    # Ví dụ:
    #     [[IMAGE_1]]
    #
    placeholder: str = Field(
        ...,
        description="e.g. [[IMAGE_1]]",
    )

    # Tên file image local nếu muốn lưu metadata/cache.
    filename: str = Field(
        ...,
        description="Save under images/, e.g. self_attention.jpg",
    )

    alt: str

    caption: str

    # Thay vì prompt gửi Image Model,
    # bây giờ đây là từ khóa dùng để tìm ảnh trên web.
    image_query: str = Field(
        ...,
        description="Specific web image search query.",
    )

    # Các field dưới đây được điền sau khi search.
    image_url: Optional[str] = None
    source_url: Optional[str] = None

    # Nguồn ảnh / license nếu API tìm được.
    credit: Optional[str] = None


class GlobalImagePlan(BaseModel):

    # Markdown sau khi LLM chèn [[IMAGE_1]], ...
    md_with_placeholders: str

    # Danh sách image cần generate
    images: List[ImageSpec] = Field(
        default_factory=list
    )


# ============================================================
# 3. LANGGRAPH STATE
# ============================================================
#
# State là "bộ nhớ dùng chung" của workflow.
#
# Có thể hình dung:
#
#                 STATE
#                   │
#       ┌───────────┼────────────┐
#       ▼           ▼            ▼
#    Router      Planner       Worker
#       │           │            │
#       └───────────┼────────────┘
#                   ▼
#                Reducer
#
# ============================================================

class State(TypedDict):

    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    topic: str

    # Ngôn ngữ đầu ra được xác định từ topic của người dùng.
    # Mục tiêu là giữ toàn bộ workflow nhất quán về ngôn ngữ.
    language: str

    # --------------------------------------------------------
    # Router / Research
    # --------------------------------------------------------

    mode: str

    needs_research: bool

    queries: List[str]

    evidence: List[EvidenceItem]

    # --------------------------------------------------------
    # Planner
    # --------------------------------------------------------

    plan: Optional[Plan]

    # --------------------------------------------------------
    # Workers
    # --------------------------------------------------------
    #
    # Mỗi worker trả:
    #
    #     (task_id, section_markdown)
    #
    # operator.add giúp LangGraph MERGE kết quả
    # từ nhiều worker lại.
    #
    # Ví dụ:
    #
    # Worker 1 -> [(1, "...")]
    # Worker 2 -> [(2, "...")]
    # Worker 3 -> [(3, "...")]
    #
    # ↓
    #
    # sections =
    # [
    #     (1, "..."),
    #     (2, "..."),
    #     (3, "...")
    # ]
    #
    # --------------------------------------------------------

    sections: Annotated[
        List[tuple[int, str]],
        operator.add
    ]

    # --------------------------------------------------------
    # Reducer / Images
    # --------------------------------------------------------

    merged_md: str

    md_with_placeholders: str

    image_specs: List[dict]

    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    final: str


# ============================================================
# 4. LLM
# ============================================================

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY is missing. "
        "Please add it to .env"
    )

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY is missing. "
        "Please add it to .env"
    )


GEMINI_MAIN_MODEL = os.getenv(
    "GEMINI_MAIN_MODEL",
    "gemini-3.1-flash-lite",
)

GEMINI_FAST_MODEL = os.getenv(
    "GEMINI_FAST_MODEL",
    "gemini-3.5-flash-lite",
)

GROQ_WORKER_MODEL = os.getenv(
    "GROQ_WORKER_MODEL",
    "openai/gpt-oss-120b",
)


# ------------------------------------------------------------
# Gemini = model chính
# ------------------------------------------------------------

gemini_main = ChatGoogleGenerativeAI(
    model=GEMINI_MAIN_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0,
)


# ------------------------------------------------------------
# Gemini Flash-Lite = task nhẹ
# ------------------------------------------------------------

gemini_fast = ChatGoogleGenerativeAI(
    model=GEMINI_FAST_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0,
)


# ------------------------------------------------------------
# Groq = Worker writer
# ------------------------------------------------------------

groq_worker = ChatGroq(
    model=GROQ_WORKER_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
)


print("\n========== MODELS ==========")
print("Gemini main :", GEMINI_MAIN_MODEL)
print("Gemini fast :", GEMINI_FAST_MODEL)
print("Groq worker :", GROQ_WORKER_MODEL)


# ============================================================
# 5. ROUTER
# ============================================================
#
# Router trả lời câu hỏi:
#
#     "Topic này có cần web research không?"
#
# Có 3 mode:
#
# closed_book
#     ↓
# Không cần web.
#
# hybrid
#     ↓
# Có kiến thức nền + một phần thông tin mới.
#
# open_book
#     ↓
# Phụ thuộc mạnh vào thông tin mới nhất.
#
# ============================================================

ROUTER_SYSTEM = """
    You are a routing module for a technical blog planner.

    Decide whether web research is needed BEFORE planning.

    Modes:

    - closed_book (needs_research=false):
    Evergreen topics where correctness does not depend on
    recent facts (concepts, fundamentals).

    - hybrid (needs_research=true):
    Mostly evergreen but needs up-to-date examples/tools/models
    to be useful.

    - open_book (needs_research=true):
    Mostly volatile: weekly roundups, "this week", "latest",
    rankings, pricing, policy/regulation.

    Language:

    - The final article language is provided separately by the caller.
    - Do NOT let the language of web sources determine the final article language.

    If needs_research=true:

    - Output 3–10 high-signal queries.
    - Queries should be scoped and specific.
    - Avoid generic queries like just "AI" or "LLM".
    - If user asked for "last week/this week/latest",
    reflect that constraint IN THE QUERIES.
"""


def router_node(state: State) -> dict:

    topic = state["topic"]

    # Structured output giúp LLM trả đúng RouterDecision
    # Router chỉ là task nhẹ nên giao cho Gemini Flash-Lite.
    decider = gemini_fast.with_structured_output(
        RouterDecision
    )

    decision = decider.invoke(
        [
            SystemMessage(
                content=ROUTER_SYSTEM
            ),
            HumanMessage(
                content=(
                    f"Topic: {topic}\n"
                    f"Final output language: {state.get('language', 'English')}"
                )
            ),
        ]
    )

    print("\n========== ROUTER ==========")
    print(decision.model_dump())

    return {
        "needs_research": decision.needs_research,
        "mode": decision.mode,
        "queries": decision.queries,
    }


# ------------------------------------------------------------
# Router conditional edge
# ------------------------------------------------------------

def route_next(state: State) -> str:

    if state["needs_research"]:
        return "research"

    return "orchestrator"


# ============================================================
# 6. TAVILY RESEARCH
# ============================================================

def _tavily_search(
    query: str,
    max_results: int = 5,
) -> List[dict]:
    """
    Chạy một Tavily search và normalize kết quả.

    Normalize rất quan trọng vì:
    raw result của tool có thể có format khác nhau.

    Ta chuyển tất cả về format thống nhất.
    """

    tool = TavilySearchResults(
        max_results=max_results
    )

    results = tool.invoke(
        {
            "query": query
        }
    )

    normalized: List[dict] = []

    for r in results or []:

        normalized.append(
            {
                "title": r.get("title") or "",
                "url": r.get("url") or "",
                "snippet": (
                    r.get("content")
                    or r.get("snippet")
                    or ""
                ),
                "published_at": (
                    r.get("published_date")
                    or r.get("published_at")
                ),
                "source": r.get("source"),
            }
        )

    return normalized


# ============================================================
# 7. RESEARCH SYNTHESIZER
# ============================================================

RESEARCH_SYSTEM = """
    You are a research synthesizer for technical writing.

    Given raw web search results, produce a deduplicated
    list of EvidenceItem objects.

    Rules:

    - Only include items with a non-empty url.
    - Prefer relevant + authoritative sources.
    - If a published date is explicitly present,
    keep it as YYYY-MM-DD.
    - If missing or unclear, set published_at=null.
    - Do NOT guess dates.
    - Keep snippets short.
    - Deduplicate by URL.
    - Evidence may remain in its original language.
      The final article language is enforced later by the writer.
"""


def research_node(state: State) -> dict:

    queries = (
        state.get("queries", [])
        or []
    )

    max_results = 6

    raw_results: List[dict] = []

    # --------------------------------------------------------
    # Chạy các query song song.
    #
    # Trước đây:
    #
    #     query 1 → search
    #     query 2 → search
    #     query 3 → search
    #
    # Bây giờ dùng ThreadPoolExecutor để các request I/O
    # tới Tavily có thể chạy đồng thời.
    #
    # Đây là một tối ưu latency quan trọng vì research
    # không cần phải chờ query trước hoàn thành.
    # --------------------------------------------------------

    if queries:

        with ThreadPoolExecutor(
            max_workers=min(5, len(queries))
        ) as executor:

            futures = {
                executor.submit(
                    _tavily_search,
                    query,
                    max_results,
                ): query
                for query in queries
            }

            for future in as_completed(futures):

                query = futures[future]

                try:
                    raw_results.extend(
                        future.result()
                    )

                except Exception as e:

                    print(
                        f"⚠️ Research failed for query "
                        f"'{query}': {e}"
                    )

    # Không có kết quả
    if not raw_results:

        print("\n⚠️ No research results.")

        return {
            "evidence": []
        }

    print(
        f"\n🔎 Raw research results: "
        f"{len(raw_results)}"
    )

    # --------------------------------------------------------
    # LLM tổng hợp evidence
    #
    # Đây là task structured-output tương đối nhẹ,
    # nên giao cho Gemini Flash-Lite thay vì model chính.
    # --------------------------------------------------------

    extractor = gemini_fast.with_structured_output(
        EvidencePack
    )

    pack = extractor.invoke(
        [
            SystemMessage(
                content=RESEARCH_SYSTEM
            ),

            HumanMessage(
                content=(
                    f"Raw results:\n"
                    f"{raw_results}"
                )
            ),
        ]
    )

    # --------------------------------------------------------
    # Deduplicate theo URL
    # --------------------------------------------------------

    dedup = {}

    for evidence in pack.evidence:

        if evidence.url:
            dedup[evidence.url] = evidence

    evidence = list(
        dedup.values()
    )

    print(
        f"✅ Evidence after dedup: "
        f"{len(evidence)}"
    )

    return {
        "evidence": evidence
    }


# ============================================================
# 8. ORCHESTRATOR / PLANNER
# ============================================================
#
# Router:
#
#     "Có cần research không?"
#
# Research:
#
#     "Đây là evidence."
#
# Orchestrator:
#
#     "Dựa vào topic + mode + evidence,
#      hãy lập kế hoạch."
#
# ============================================================

ORCH_SYSTEM = """
    You are a senior technical writer and developer advocate.

    Your job is to produce a highly actionable outline
    for a technical blog post.

    Hard requirements:

    - Create 5–9 sections (tasks).
    - Each task must include:
    1) goal
    2) 3–6 bullets
    3) target word count (120–550)

    Quality bar:

    - Assume the reader is a developer.
    - Use correct terminology.
    - Bullets must be actionable:
    build/compare/measure/verify/debug.

    Ensure the overall plan includes at least 2 of:

    - minimal code sketch / MWE
    - edge cases / failure modes
    - performance/cost considerations
    - security/privacy considerations
    - debugging/observability tips

    Grounding rules:

    - closed_book:
    keep it evergreen.

    - hybrid:
    use evidence for up-to-date examples.
    Mark fresh sections with:
        requires_research=True
        requires_citations=True

    - open_book:
    set blog_kind="news_roundup".
    Every section should summarize events + implications.
    Do NOT create tutorial sections unless explicitly requested.

    If evidence is insufficient:
    transparently say "insufficient sources".

    Language rule:
    - Write the blog title, section titles, goals, bullets,
      and all other prose fields in the requested output language.
    - The requested output language is provided by the caller.
    - Technical terms may keep standard English names when appropriate.

    Output must strictly match the Plan schema.
"""


def orchestrator_node(state: State) -> dict:

    # Planner là một trong những node quan trọng nhất,
    # nên dùng Gemini model chính.
    planner = gemini_main.with_structured_output(
        Plan
    )

    evidence = state.get(
        "evidence",
        []
    )

    mode = state.get(
        "mode",
        "closed_book"
    )

    evidence_data = [
        e.model_dump()
        for e in evidence
    ]

    plan = planner.invoke(
        [
            SystemMessage(
                content=ORCH_SYSTEM
            ),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Output language: {state.get('language', 'English')}\n"
                    f"Mode: {mode}\n\n"
                    f"Evidence "
                    f"(ONLY use for fresh claims; "
                    f"may be empty):\n"
                    f"{evidence_data[:16]}"
                )
            ),
        ]
    )

    print("\n========== PLAN ==========")
    print(
        plan.model_dump_json(
            indent=2
        )
    )

    return {
        "plan": plan
    }


# ============================================================
# 9. FAN-OUT
# ============================================================
#
# Đây là một phần rất quan trọng của LangGraph.
#
# Plan:
#
#     Task 1
#     Task 2
#     Task 3
#     Task 4
#
# fanout biến nó thành:
#
#     Worker(Task 1)
#     Worker(Task 2)
#     Worker(Task 3)
#     Worker(Task 4)
#
# Các worker có thể chạy song song.
#
# ============================================================

def fanout(state: State):

    plan = state["plan"]

    return [
        Send(
            "worker",
            {
                "task": task.model_dump(),

                "topic": state["topic"],

                "mode": state["mode"],

                # Truyền ngôn ngữ xuống từng Worker để tránh language drift.
                "language": state.get("language", "English"),

                "plan": plan.model_dump(),

                "evidence": [
                    e.model_dump()
                    for e in state.get(
                        "evidence",
                        []
                    )
                ],
            },
        )

        for task in plan.tasks
    ]


# ============================================================
# 10. WORKER
# ============================================================
#
# Mỗi worker chỉ chịu trách nhiệm:
#
#     1 section
#
# Không viết toàn bộ blog.
#
# Đây là tư tưởng:
#
#     Planner → phân chia công việc
#     Worker  → thực thi từng phần
#
# ============================================================

WORKER_SYSTEM = """
You are a senior technical writer and developer advocate.

Write ONE section of a technical blog post in Markdown.

Hard constraints:

- Follow the provided Goal.
- Cover ALL Bullets in order.
- Stay close to Target words (±15%).
- Output ONLY the section content in Markdown.
- Do NOT output blog title H1.
- Start with:
  ## <Section Title>

Scope guard:

- If blog_kind == "news_roundup":
  do NOT turn this into a tutorial.
- Focus on summarizing events and implications.

Grounding policy:

- If mode == open_book:
  specific event/company/model/funding/policy claims
  MUST be supported by provided Evidence URLs.

- For each event claim:
  attach a Markdown source link.

- Only use URLs provided in Evidence.

- If information is not supported:
  write:
  "Not found in provided sources."

- If requires_citations == true:
  cite Evidence URLs.

- Evergreen reasoning is okay without citations
  unless requires_citations=true.

Code:

- If requires_code == true:
  include at least one minimal,
  correct code snippet.

Style:

- Short paragraphs.
- Bullets where useful.
- Code fences for code.
- Avoid fluff/marketing.
- Be precise and implementation-oriented.
"""


def worker_node(payload: dict) -> dict:

    # --------------------------------------------------------
    # Deserialize payload
    #
    # Send() truyền dictionary,
    # nên ta convert ngược về Pydantic model.
    # --------------------------------------------------------

    task = Task(
        **payload["task"]
    )

    plan = Plan(
        **payload["plan"]
    )

    evidence = [
        EvidenceItem(**e)
        for e in payload.get(
            "evidence",
            []
        )
    ]

    topic = payload["topic"]

    # Ngôn ngữ được truyền từ State xuống Worker.
    language = payload.get(
        "language",
        "English"
    )

    mode = payload.get(
        "mode",
        "closed_book"
    )

    # --------------------------------------------------------
    # Format bullets
    # --------------------------------------------------------

    bullets_text = (
        "\n- "
        + "\n- ".join(task.bullets)
    )

    # --------------------------------------------------------
    # Format evidence
    # --------------------------------------------------------

    evidence_text = ""

    if evidence:

        evidence_text = "\n".join(
            (
                f"- {e.title} | "
                f"{e.url} | "
                f"{e.published_at or 'date:unknown'}"
            ).strip()

            for e in evidence[:20]
        )

    # --------------------------------------------------------
    # Generate section
    # --------------------------------------------------------

    # Worker có nhiều task nhỏ và chạy fan-out,
    # nên giao cho Groq vì model này có tốc độ token rất cao.
    response = groq_worker.invoke(
        [
            SystemMessage(
                content=WORKER_SYSTEM
            ),

            HumanMessage(
                content=(
                    f"Blog title: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Constraints: {plan.constraints}\n"
                    f"Topic: {topic}\n"
                    f"Output language: {language}\n"
                    f"Mode: {mode}\n\n"

                    f"Section title: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Tags: {task.tags}\n"
                    f"requires_research: "
                    f"{task.requires_research}\n"
                    f"requires_citations: "
                    f"{task.requires_citations}\n"
                    f"requires_code: "
                    f"{task.requires_code}\n"

                    f"Bullets:"
                    f"{bullets_text}\n\n"

                    "Evidence "
                    "(ONLY use these URLs when citing):\n"
                    f"{evidence_text}\n"
                )
            ),
        ]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # AIMessage.content thường là str,
    # nhưng một số provider có thể trả list.
    #
    # Ta normalize để worker không bị:
    #
    #     AttributeError:
    #     'list' object has no attribute 'strip'
    #
    # --------------------------------------------------------

    content = response.content

    if isinstance(content, list):

        section_md = "".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    else:

        section_md = content

    section_md = section_md.strip()

    print(
        f"\n✍️ Worker finished: "
        f"Task {task.id} - {task.title}"
    )

    # --------------------------------------------------------
    # QUAN TRỌNG:
    #
    # task.id được trả về cùng section.
    #
    # Reducer sau đó sort theo task.id
    # để đảm bảo thứ tự blog không bị đảo.
    # --------------------------------------------------------

    return {
        "sections": [
            (
                task.id,
                section_md
            )
        ]
    }


# ============================================================
# 11. REDUCER SUBGRAPH
# ============================================================
#
# Đây là điểm nâng cấp khá lớn của project.
#
# Reducer không còn chỉ là:
#
#     worker → reducer → END
#
# Mà trở thành một SUBGRAPH:
#
#                 reducer
#                    │
#                    ▼
#              merge_content
#                    │
#                    ▼
#              decide_images
#                    │
#                    ▼
#        generate_and_place_images
#
# ============================================================


# ============================================================
# 11.1 MERGE CONTENT
# ============================================================

def merge_content(state: State) -> dict:

    plan = state["plan"]

    # --------------------------------------------------------
    # Worker chạy song song nên thứ tự result
    # không nhất thiết giống Task ID.
    #
    # Ví dụ:
    #
    # [(3, "..."), (1, "..."), (2, "...")]
    #
    # Sort lại:
    #
    # [(1, "..."), (2, "..."), (3, "...")]
    # --------------------------------------------------------

    ordered_sections = [
        md
        for _, md in sorted(
            state["sections"],
            key=lambda x: x[0]
        )
    ]

    body = "\n\n".join(
        ordered_sections
    ).strip()

    merged_md = (
        f"# {plan.blog_title}\n\n"
        f"{body}\n"
    )

    return {
        "merged_md": merged_md
    }


# ============================================================
# 11.2 DECIDE IMAGES
# ============================================================

DECIDE_IMAGES_SYSTEM = """
You are an expert technical editor.

Decide if images/diagrams are needed for THIS blog.

Rules:

- Maximum 3 images total.
- Each image must materially improve understanding.
- Prefer diagrams/flows/technical visuals.
- Avoid decorative images.
- Insert placeholders exactly:
  [[IMAGE_1]]
  [[IMAGE_2]]
  [[IMAGE_3]]

IMPORTANT:
- We do NOT generate images with an image model.
- Instead, propose a specific web image search query.
- Prefer queries that are likely to find educational,
  technical, Wikimedia Commons, documentation, or openly
  licensed visuals.
- The image_query must be specific, not generic.

For every proposed image:
- placeholder
- filename
- alt
- caption
- image_query

If no images are needed:

    md_with_placeholders = input

    images = []

Return strictly GlobalImagePlan.
"""


def decide_images(state: State) -> dict:

    # Image planning là task nhẹ,
    # nên dùng Gemini Flash-Lite.
    planner = gemini_fast.with_structured_output(
        GlobalImagePlan
    )

    merged_md = state["merged_md"]

    plan = state["plan"]

    assert plan is not None

    image_plan = planner.invoke(
        [
            SystemMessage(
                content=DECIDE_IMAGES_SYSTEM
            ),

            HumanMessage(
                content=(
                    f"Blog kind: "
                    f"{plan.blog_kind}\n"
                    f"Topic: "
                    f"{state['topic']}\n"
                    f"Output language: "
                    f"{state.get('language', 'English')}\n\n"

                    "Insert placeholders + "
                    "propose web image search queries.\n\n"

                    f"{merged_md}"
                )
            ),
        ]
    )

    print("\n========== IMAGE PLAN ==========")

    print(
        image_plan.model_dump_json(
            indent=2
        )
    )

    return {
        "md_with_placeholders": (
            image_plan.md_with_placeholders
        ),

        "image_specs": [
            image.model_dump()
            for image in image_plan.images
        ],
    }


# ============================================================
# 11.3 WEB IMAGE SEARCH
# ============================================================
#
# Trước đây project dùng:
#
#     Gemini Image Model
#             ↓
#        generate image
#
# Nhưng image generation có thể ăn quota khá nhanh.
#
# Phiên bản này:
#
#     Gemini
#        ↓
#     image_query
#        ↓
#     Wikimedia Commons API
#        ↓
#     image URL
#        ↓
#     Markdown
#
# Ưu điểm:
#     - Không tốn image-generation quota.
#     - Không cần thêm image API key.
#     - Có source URL để người đọc kiểm tra nguồn.
#
# Lưu ý:
#     Wikimedia Commons có nhiều license khác nhau.
#     Vì vậy ta giữ lại source/credit trong Markdown
#     thay vì giả định mọi ảnh đều tự do sử dụng.
#
# ============================================================


def _wikimedia_search_image(
    query: str,
) -> Optional[dict]:
    """
    Tìm một hình ảnh trên Wikimedia Commons.

    Wikimedia API không yêu cầu API key cho kiểu search này.

    Return:
        {
            "image_url": "...",
            "source_url": "...",
            "title": "...",
            "credit": "..."
        }

    hoặc None nếu không tìm thấy.
    """

    api_url = (
        "https://commons.wikimedia.org/w/api.php"
    )

    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": query,
        "gsrnamespace": 6,
        "gsrlimit": 5,
        "prop": "imageinfo",
        "iiprop": "url|extmetadata",
        "iiurlwidth": 1400,
        "format": "json",
    }

    response = requests.get(
        api_url,
        params=params,
        timeout=20,
        headers={
            "User-Agent":
                "AgenticWriter/1.0"
        },
    )

    response.raise_for_status()

    data = response.json()

    pages = (
        data.get("query", {})
        .get("pages", {})
    )

    if not pages:
        return None

    # Chọn kết quả đầu tiên có thumbnail/image URL.
    for page in pages.values():

        imageinfo = (
            page.get("imageinfo") or []
        )

        if not imageinfo:
            continue

        info = imageinfo[0]

        image_url = (
            info.get("thumburl")
            or info.get("url")
        )

        if not image_url:
            continue

        title = page.get(
            "title",
            "Wikimedia Commons",
        )

        source_url = (
            "https://commons.wikimedia.org/wiki/"
            + title.replace(" ", "_")
        )

        metadata = info.get(
            "extmetadata",
            {},
        )

        artist = (
            metadata.get("Artist", {})
            .get("value")
        )

        license_name = (
            metadata.get("LicenseShortName", {})
            .get("value")
        )

        credit_parts = []

        if artist:
            # Metadata đôi khi chứa HTML.
            artist_clean = re.sub(
                r"<[^>]+>",
                "",
                artist,
            ).strip()

            if artist_clean:
                credit_parts.append(
                    artist_clean
                )

        if license_name:
            credit_parts.append(
                license_name
            )

        credit = " — ".join(
            credit_parts
        ) or "Wikimedia Commons"

        return {
            "image_url": image_url,
            "source_url": source_url,
            "title": title,
            "credit": credit,
        }

    return None


# ============================================================
# 11.4 FIND + PLACE WEB IMAGES
# ============================================================

def generate_and_place_images(
    state: State,
) -> dict:

    plan = state["plan"]

    assert plan is not None

    md = (
        state.get(
            "md_with_placeholders"
        )
        or state["merged_md"]
    )

    image_specs = (
        state.get(
            "image_specs",
            []
        )
        or []
    )

    # --------------------------------------------------------
    # Nếu LLM quyết định không cần ảnh
    # --------------------------------------------------------

    if not image_specs:

        filename = (
            safe_filename(
                plan.blog_title
            )
            + ".md"
        )

        Path(filename).write_text(
            md,
            encoding="utf-8",
        )

        print(
            f"\n📄 Markdown saved: "
            f"{Path(filename).resolve()}"
        )

        return {
            "final": md
        }

    # --------------------------------------------------------
    # Tạo thư mục images/
    #
    # Hiện tại ta không bắt buộc download ảnh.
    # Folder này chỉ được giữ lại để tương thích
    # với cấu trúc project cũ.
    # --------------------------------------------------------

    images_dir = Path(
        "images"
    )

    images_dir.mkdir(
        exist_ok=True
    )

    # --------------------------------------------------------
    # Search từng image trên web.
    #
    # Tối đa 3 ảnh nên chạy tuần tự là đủ.
    # Nếu sau này tăng số lượng ảnh,
    # có thể fan-out phần này bằng ThreadPoolExecutor.
    # --------------------------------------------------------

    for spec in image_specs:

        placeholder = (
            spec["placeholder"]
        )

        query = (
            spec.get("image_query")
            or spec.get("alt")
            or plan.blog_title
        )

        try:

            result = (
                _wikimedia_search_image(
                    query
                )
            )

        except Exception as e:

            result = None

            print(
                f"⚠️ Image search failed "
                f"for '{query}': {e}"
            )

        # ----------------------------------------------------
        # Nếu không tìm thấy ảnh:
        #
        # Blog vẫn được tạo.
        # Không để image search làm chết workflow.
        # ----------------------------------------------------

        if not result:

            fallback = (
                "> **[IMAGE NOT FOUND]** "
                f"{spec.get('caption', '')}\n"
                ">\n"
                f"> **Search:** {query}\n"
            )

            md = md.replace(
                placeholder,
                fallback,
            )

            continue

        image_url = result["image_url"]
        source_url = result["source_url"]
        credit = result["credit"]

        # ----------------------------------------------------
        # Lưu thông tin vào spec để debug / inspect state.
        # ----------------------------------------------------

        spec["image_url"] = image_url
        spec["source_url"] = source_url
        spec["credit"] = credit

        # ----------------------------------------------------
        # Thay placeholder bằng Markdown image.
        #
        # Ảnh được load trực tiếp từ URL trên web.
        # Không cần Gemini image generation.
        # ----------------------------------------------------

        img_md = (
            f"![{spec['alt']}]"
            f"({image_url})\n"
            f"*{spec['caption']}*\n"
            f"\n"
            f"*Source: [{credit}]({source_url})*"
        )

        md = md.replace(
            placeholder,
            img_md,
        )

        print(
            f"🖼️ Image found: "
            f"{spec.get('filename', query)}"
        )

    # --------------------------------------------------------
    # Save final Markdown
    # --------------------------------------------------------

    filename = (
        safe_filename(
            plan.blog_title
        )
        + ".md"
    )

    output_path = Path(
        filename
    )

    output_path.write_text(
        md,
        encoding="utf-8",
    )

    print(
        f"\n📄 Final Markdown saved:"
        f"\n{output_path.resolve()}"
    )

    return {
        "final": md
    }


# ============================================================
# 12. SAFE FILENAME
# ============================================================
#
# Đây là phần mình cố tình thêm.
#
# Trước đây title có thể là:
#
#     Understanding Machine Learning:
#     A Beginner's Roadmap
#
# Windows không cho phép ':' trong filename.
#
# Vì vậy phải sanitize title trước khi tạo file.
#
# ============================================================

def safe_filename(
    title: str,
) -> str:

    # Xóa ký tự không hợp lệ trên Windows
    filename = re.sub(
        r'[<>:"/\\|?*]',
        "",
        title,
    )

    # Thay nhiều khoảng trắng bằng _
    filename = re.sub(
        r"\s+",
        "_",
        filename,
    )

    return filename.lower()


# ============================================================
# 13. BUILD REDUCER SUBGRAPH
# ============================================================
#
# Subgraph:
#
# START
#   ↓
# merge_content
#   ↓
# decide_images
#   ↓
# generate_and_place_images
#   ↓
# END
#
# ============================================================

reducer_graph = StateGraph(
    State
)

reducer_graph.add_node(
    "merge_content",
    merge_content,
)

reducer_graph.add_node(
    "decide_images",
    decide_images,
)

reducer_graph.add_node(
    "generate_and_place_images",
    generate_and_place_images,
)

reducer_graph.add_edge(
    START,
    "merge_content",
)

reducer_graph.add_edge(
    "merge_content",
    "decide_images",
)

reducer_graph.add_edge(
    "decide_images",
    "generate_and_place_images",
)

reducer_graph.add_edge(
    "generate_and_place_images",
    END,
)

reducer_subgraph = (
    reducer_graph.compile()
)


# ============================================================
# 14. MAIN GRAPH
# ============================================================
#
# Đây là toàn bộ Agentic Writer:
#
#
#                     START
#                       │
#                       ▼
#                    ROUTER
#                  /         \
#                 /           \
#                ▼             ▼
#           RESEARCH       ORCHESTRATOR
#                │             │
#                └──────┬──────┘
#                       ▼
#                  ORCHESTRATOR
#                       │
#                     FANOUT
#                  /    |    \
#                 ▼     ▼     ▼
#              Worker Worker Worker
#                 \     |     /
#                  \    |    /
#                    REDUCER
#                       │
#                ┌──────┴──────┐
#                ▼             ▼
#             Merge      Web Images
#                │             │
#                └──────┬──────┘
#                       ▼
#                      END
#
# ============================================================

g = StateGraph(
    State
)

# ------------------------------------------------------------
# Main nodes
# ------------------------------------------------------------

g.add_node(
    "router",
    router_node,
)

g.add_node(
    "research",
    research_node,
)

g.add_node(
    "orchestrator",
    orchestrator_node,
)

g.add_node(
    "worker",
    worker_node,
)

# Subgraph được gắn như một node
g.add_node(
    "reducer",
    reducer_subgraph,
)


# ------------------------------------------------------------
# Main edges
# ------------------------------------------------------------

g.add_edge(
    START,
    "router",
)

# Router quyết định:
#
# research
#     hoặc
#
# orchestrator
#
g.add_conditional_edges(
    "router",
    route_next,
    {
        "research": "research",
        "orchestrator": "orchestrator",
    },
)

# Research xong → Planner
g.add_edge(
    "research",
    "orchestrator",
)

# Planner → Fan-out Workers
g.add_conditional_edges(
    "orchestrator",
    fanout,
    ["worker"],
)

# Worker → Reducer Subgraph
g.add_edge(
    "worker",
    "reducer",
)

# Reducer → END
g.add_edge(
    "reducer",
    END,
)


# ============================================================
# 15. POSTGRES CHECKPOINTER
# ============================================================
#
# Đây là nâng cấp lớn so với InMemorySaver.
#
# InMemorySaver:
#
#     RAM
#      ↓
#     restart app
#      ↓
#     mất state
#
#
# PostgreSQL:
#
#     LangGraph
#         ↓
#     PostgreSQL
#         ↓
#     restart app
#         ↓
#     checkpoint vẫn còn
#
# ============================================================

DATABASE_URL = (
    get_database_url()
)

_conn = psycopg.connect(
    DATABASE_URL,
    autocommit=True,
    row_factory=dict_row,
)

checkpointer = PostgresSaver(
    _conn
)

# Tạo các bảng checkpoint nếu chưa tồn tại
checkpointer.setup()


# ------------------------------------------------------------
# Compile final application
# ------------------------------------------------------------

app = g.compile(
    checkpointer=checkpointer
)


# ============================================================
# 16. CONFIG
# ============================================================
#
# thread_id dùng để LangGraph biết:
#
#     "Đây là conversation / workflow nào?"
#
# PostgreSQL sẽ lưu checkpoint dựa trên thread này.
#
# ============================================================

config = {
    "configurable": {
        "thread_id": "test_thread_id_2"
    }
}


# ============================================================
# 17. RUNNER
# ============================================================

def detect_output_language(text: str) -> str:
    """
    Xác định ngôn ngữ đầu ra dựa trên input của người dùng.

    Với workflow hiện tại, ta ưu tiên tiếng Việt khi input có
    dấu/chữ đặc trưng của tiếng Việt. Nếu không, mặc định tiếng Anh.

    Không gọi thêm LLM cho bước này để tránh tăng latency và quota.
    """

    vietnamese_chars = set(
        "ăâđêôơưáàảãạắằẳẵặấầẩẫậ"
        "éèẻẽẹếềểễệíìỉĩị"
        "óòỏõọốồổỗộớờởỡợ"
        "úùủũụứừửữựýỳỷỹỵ"
    )

    normalized = text.lower()

    if any(char in vietnamese_chars for char in normalized):
        return "Vietnamese"

    return "English"


def run(
    topic: str,
    as_of: Optional[str] = None,
):

    # Nếu caller không truyền ngày,
    # lấy ngày hiện tại.
    if as_of is None:
        as_of = date.today().isoformat()

    # --------------------------------------------------------
    # Giữ folder images/ cho tương lai.
    #
    # Phiên bản hiện tại chỉ nhúng URL ảnh từ Wikimedia vào Markdown,
    # nên folder này chưa dùng để lưu file ảnh.
    # mkdir(exist_ok=True) đảm bảo:
    #     - Mỗi lần run folder được tạo nếu chưa có.
    #     - Nếu folder đã tồn tại thì không làm gì thêm.
    # --------------------------------------------------------
    Path("images").mkdir(
        parents=True,
        exist_ok=True,
    )

    # Xác định ngôn ngữ đầu ra một lần ở đầu workflow.
    # Sau đó truyền xuống Planner/Worker/Image planner để tránh lẫn ngôn ngữ.
    language = detect_output_language(topic)

    # --------------------------------------------------------
    # IMPORTANT
    #
    # State hiện tại không dùng as_of.
    #
    # Vì vậy KHÔNG truyền as_of vào app.invoke()
    # cho tới khi bạn thực sự thêm nó vào State.
    # --------------------------------------------------------

    initial_state = {

        "topic": topic,

        # Ngôn ngữ được giữ xuyên suốt toàn bộ graph.
        "language": language,

        "mode": "",

        "needs_research": False,

        "queries": [],

        "evidence": [],

        "plan": None,

        "sections": [],

        "merged_md": "",

        "md_with_placeholders": "",

        "image_specs": [],

        "final": "",
    }

    out = app.invoke(
        initial_state,
        config=config,
    )

    return out


# ============================================================
# 18. TEST
# ============================================================

result = run(
    "Giải thích kiến trúc Transformer và cho biết những cải tiến đáng chú ý của các phiên bản Transformer gần đây."
)

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)

print(
    result.get(
        "final",
        ""
    )
)

C:\Users\vothi\AppData\Local\Temp\ipykernel_17452\445170503.py:27: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults



========== MODELS ==========
Gemini main : gemini-3.1-flash-lite
Gemini fast : gemini-3.5-flash-lite
Groq worker : openai/gpt-oss-120b


Deserializing unregistered type __main__.Plan from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'Plan')]
Deserializing unregistered type __main__.EvidenceItem from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'EvidenceItem')]
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



========== ROUTER ==========
{'needs_research': True, 'mode': 'hybrid', 'queries': ['recent transformer architecture variants improvements 2023 2024', 'state of the art transformer efficiency improvements flashattention', 'linear attention mechanisms transformer architecture updates', 'mixture of experts transformer architectural improvements']}


C:\Users\vothi\AppData\Local\Temp\ipykernel_17452\445170503.py:631: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(



🔎 Raw research results: 24
✅ Evidence after dedup: 14

========== PLAN ==========
{
  "blog_title": "Giải mã kiến trúc Transformer: Từ nền tảng đến các cải tiến đột phá",
  "audience": "Các kỹ sư phần mềm, nhà phát triển AI và những người quan tâm đến kiến trúc mô hình ngôn ngữ lớn (LLM).",
  "tone": "Chuyên nghiệp, kỹ thuật, súc tích.",
  "blog_kind": "explainer",
  "constraints": [
    "Sử dụng thuật ngữ kỹ thuật chính xác.",
    "Đảm bảo tính cập nhật với các nghiên cứu mới nhất.",
    "Cấu trúc bài viết logic từ cơ bản đến nâng cao."
  ],
  "tasks": [
    {
      "id": 1,
      "title": "Kiến trúc Transformer nguyên bản: Cơ chế Self-Attention",
      "goal": "Người đọc hiểu được tại sao cơ chế Self-Attention lại là trái tim của Transformer và hạn chế về độ phức tạp tính toán.",
      "bullets": [
        "Phân tích cơ chế Scaled Dot-Product Attention.",
        "Giải thích tại sao độ phức tạp O(n²) của attention là nút thắt cổ chai.",
        "Mô tả vai trò của Encoder và Decoder 